<a href="https://colab.research.google.com/github/jayfordamancio-debug/FUNDAI-Laboratories-AMANCIO/blob/main/Lab7_Intro_to_ML_Amancio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7: Supervised vs. Unsupervised Learning

## Fundamentals of Artificial Intelligence

**Name:** Amancio
**Course:** BSCS-Ai
**Section:** 2A
**Date:** 10/22/2026  

**Dataset Source (GitHub raw URL):**  
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv  

**GitHub URL:** https://github.com/jayfordamancio-debug

## Description
This laboratory compares supervised and unsupervised learning, demonstrates overfitting and underfitting using decision trees, and evaluates models using k-fold cross-validation.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    adjusted_rand_score,
)

GITHUB_RAW_URL = "https://github.com/jayfordamancio-debug

df = pd.read_csv(GITHUB_RAW_URL)

print("Dataset shape:", df.shape)
print()
print(df.head())
print()
print("Class distribution:")
print(df["species"].value_counts())

feature_names = {
    "sepal_length" "Sepal Length",
    "sepal_width": "Sepal Width",
    "petal_length": "Petal Length",
    "petal_width": "Petal Width",
}

X = df.drop(columns=["species"])
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)



## Part A: Supervised Learning Pipeline

The model pipeline includes feature scaling using `StandardScaler` followed by a `DecisionTreeClassifier`.
Scaling is fit strictly on training data (or inside training folds during cross-validation) to prevent data leakage.
This prevents target leakage.

In [ ]:
supervised_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(random_state=42)),
    ]
)

supervised_pipeline.fit(X_train, y_train)

y_pred = supervised_pipeline.predict(X_test)

print("SUPERVISED LEARNING RESULTS")
print("=" * 50)
print("Cluster vs. True Species crosstab:")
print(comparison)
print()
print("Adjusted Rand Index (cluster agreement with true labels):")
print(adjusted_rand_score(y, cluster_labels))

# Step 10: Visualize Clusters with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', edgecolor='k')
plt.title("K-Means Clusters (PCA-reduced)")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.subplot(1, 2, 2)
species_codes = pd.factorize(y) [0]
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k')
plt.title("True Species (PCA-reduced)")
plt.xlabel("PC1")

plt.tight_layout()
plt.show()

print("Explained variance ratio:" , pca.explained_variance_ration_)



## Part C: Complexity Sweep (Underfitting vs. Overfitting)

We train decision tree classifiers with varying `max_depth` values to observe the transition:
- Very small depth (underfitting / high bias)
- Moderate depth (good fit / balanced)
- Very large depth (overfitting / high variance)



In [ ]:
# Step 12: Run the Sweep
depth_values = range(1, 16)
train_scores = []
test_scores = []

for depth in depth_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=depth, random_state=42))
    ])

    pipe.fit(X_train, y_train)

    train_scores.append(pipe.score(X_train, y_train))
    test_scores.append(pipe.score(X_test, y_test))

sweep_table = pd.DataFrame({
    "max_depth": depth_values,
    "train_accuracy": train_scores,
    "test_accuracy": test_scores,
    "gap": np.array(train_scores) - np.array(test_scores)
})

print(sweep_table.to_string(index=False))

# Step 13: Plot the Bias–Variance Curve
plt.figure(figsize=(9, 5))

plt.plot(depth_values, train_scores, label="Train Accuracy", marker="o")
plt.plot(depth_values, test_scores, label="Test Accuracy", marker="s")

plt.axvspan(1, 2, alpha=0.15, color="red", label="Underfitting Region")
plt.axvspan(3, 6, alpha=0.15, color="green", label="Sweet Spot Region")
plt.axvspan(7, 15, alpha=0.15, color="orange", label="Overfitting Region")

plt.xlabel("Tree max_depth")
plt.ylabel("Accuracy")
plt.title("Bias–Variance Tradeoff Sweep (Decision Tree)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Observed Regions (fill based on your table):

- **Underfitting depth(s):** [e.g., 1–2] — Both train and test accuracy are low.
- **Sweet-spot depth(s):** [e.g., 3–6] — Test accuracy peaks and train-test gap is small.
- **Overfitting depth(s):** [e.g., 7+] — Train accuracy approaches 100% while the train–test gap widens.

## Part D: 5-Fold Stratified Cross-Validation

Cross-validation gives a more reliable estimate than a single train/test split.
The held-out test set remains completely untouched until final reporting.
Mean ± standard deviation across folds indicates model stability.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Shallow tree (depth=1, high bias)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=1, random_state=42))
    ]),
    "Moderate tree (depth=4)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=4, random_state=42))
    ]),
    "Full tree (no depth limit, high variance)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=None, random_state=42))
    ]),
}

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")

    cv_results.append({
        "Model": name,
        "CV mean": scores.mean(),
        "CV std": scores.std(),
        "Fold scores": np.round(scores, 3)
    })

    print(name)
    print(" Fold scores:", np.round(scores, 3))
    print(f"  Mean ± std : {scores.mean():.3f} ± {scores.std():.3f}")
    print("-" * 60)

cv_table = pd.DataFrame(cv_results)
print(cv_table.to_string(index=False))